In [25]:
import csv

def calc_normal_force_coeff(p_upper, p_lower, /, q_dyn, *, calibration_factor=1.0):
    return round(((p_lower - p_upper) / q_dyn) * calibration_factor, 4)

def stream_tunnel_data(csv_path):
    with open(csv_path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader, None)

        for row in reader:
            id = int(row[0])
            aoa = float(row[1])
            p_upper = float(row[2])
            p_lower = float(row[3])
            q_dyn = float(row[4])

            yield (id, aoa, p_upper, p_lower, q_dyn)


def main():
    input_file = '../materials/wind_tunnel.csv'
    output_file = '../outputs/polar_reduced.csv'

    angles = []
    cn_values = []

    for (id, aoa, p_upper, p_lower, q_dyn) in stream_tunnel_data(input_file):
        angles.append(aoa)
        cn_values.append(calc_normal_force_coeff(p_upper, p_lower, q_dyn, calibration_factor=1.02))

    print(f'{'Angles:':<{10}}Cns:')
    for (a, cn) in zip(angles, cn_values):
        print(f'{a:<{10}}{cn}')

    polar_dict = {aoa: cn for aoa, cn in zip(angles, cn_values)}

    with open(output_file, mode="w", newline="", encoding="utf-8") as out_file:
        writer = csv.writer(out_file)
        writer.writerow(["aoa_deg", "Cn"])

        for i in range(0, len(angles), 3):
            current_aoa = angles[i]
            writer.writerow([current_aoa, polar_dict[current_aoa]])

if __name__ == '__main__':
    main()

Angles:   Cns:
-2.0      -0.1428
0.0       0.1632
2.5       0.7548
4.5       1.3056
7.0       1.9788
10.0      2.652
13.5      3.06
